## 1. Imports & Configuration

In [1]:
import os, csv, time, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import v2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DATASET_ROOT = "dataset"
OUTPUT_DIR   = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_DICE           = 3

NUM_DICE_CLASSES   = 3
DICE_CLASS_NAMES   = ["1 die", "2 dice", "3 dice"]

NUM_VALUE_CLASSES  = 6
VALUE_CLASS_NAMES  = ["1 pip", "2 pips", "3 pips", "4 pips", "5 pips", "6 pips"]

VALID_COLOURS      = ["white", "red", "blue", "green", "yellow", "purple", "peach"]
NUM_COLOUR_CLASSES = len(VALID_COLOURS)
COLOUR_CLASS_NAMES = VALID_COLOURS
COLOUR_TO_IDX      = {c: i for i, c in enumerate(VALID_COLOURS)}

VALID_SIZES        = ["small", "medium", "large"]
NUM_SIZE_CLASSES   = len(VALID_SIZES)
SIZE_CLASS_NAMES   = VALID_SIZES
SIZE_TO_IDX        = {s: i for i, s in enumerate(VALID_SIZES)}

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Tasks  : count({NUM_DICE_CLASSES})  value({NUM_VALUE_CLASSES})  "
      f"colour({NUM_COLOUR_CLASSES})  size({NUM_SIZE_CLASSES})")

for split in ["train", "val", "test"]:
    p = Path(DATASET_ROOT) / split / "labels.csv"
    assert p.exists(), f"Missing: {p} — run data_generation_Finn.ipynb first."
    df = pd.read_csv(p, nrows=1)
    required = {"image", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes"}
    missing  = required - set(df.columns)
    assert not missing, f"CSV missing columns: {missing}"
print("Dataset schema valid. Ready.")


Device : cpu
PyTorch: 2.10.0
Tasks  : count(3)  value(6)  colour(7)  size(3)
Dataset schema valid. Ready.


## 2. DiceDataset — Four-Task Labels

Each sample returns four label tensors (all shape `[MAX_DICE]` except count):

| Key | Shape | Padding value | Notes |
|---|---|---|---|
| `sym_num_dice` | scalar | — | 0-indexed (0=1die, 1=2dice, 2=3dice) |
| `sym_values`   | `[3]` | `0` | die face values 1–6, stored as 0-indexed (0–5); pad=`-1` |
| `sym_colours`  | `[3]` | `-1` | colour index 0–6; pad=`-1` |
| `sym_sizes`    | `[3]` | `-1` | size index 0–2; pad=`-1` |

A unified pad of `-1` is used for all per-slot tasks so the masked loss can
apply a single `targets >= 0` mask consistently.


In [2]:
class DiceDataset(Dataset):
    VALID_SPLITS = ("train", "val", "test")

    def __init__(self, root_dir: str, split: str = "train", transform=None):
        if split not in self.VALID_SPLITS:
            raise ValueError(f"split must be one of {self.VALID_SPLITS}")

        self.image_dir = os.path.join(root_dir, split, "images")
        self.csv_path  = os.path.join(root_dir, split, "labels.csv")
        assert os.path.isdir(self.image_dir), f"Missing image dir: {self.image_dir}"
        assert os.path.isfile(self.csv_path), f"Missing CSV: {self.csv_path}"

        self.samples = []
        with open(self.csv_path, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                self.samples.append(row)
        assert len(self.samples) > 0, "No samples found in CSV."

        self.transform = transform or v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])
        print(f"DiceDataset ({split}): {len(self.samples)} samples loaded.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row      = self.samples[idx]
        img_path = os.path.join(self.image_dir, row["image"])
        image    = Image.open(img_path).convert("RGB")
        image    = self.transform(image)

        num_dice = int(row["sym_num_dice"])
        
        raw_values = list(map(int, row["sym_values"].split()))
        colours    = row["sym_colours"].split()
        sizes      = row["sym_sizes"].split()

        count_label   = num_dice - 1   

       
        value_labels  = [(v - 1) for v in raw_values]  + [-1] * (MAX_DICE - len(raw_values))
        colour_labels = [COLOUR_TO_IDX[c] for c in colours] + [-1] * (MAX_DICE - len(colours))
        size_labels   = [SIZE_TO_IDX[s]   for s in sizes]   + [-1] * (MAX_DICE - len(sizes))

        label = {
            "sym_num_dice": torch.tensor(count_label,   dtype=torch.long),
            "sym_values":   torch.tensor(value_labels,  dtype=torch.long),
            "sym_colours":  torch.tensor(colour_labels, dtype=torch.long),
            "sym_sizes":    torch.tensor(size_labels,   dtype=torch.long),
            "text":         row.get("text", ""),
        }
        return image, label


def dice_collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    labels = {
        "sym_num_dice": torch.stack([item[1]["sym_num_dice"] for item in batch]),
        "sym_values":   torch.stack([item[1]["sym_values"]   for item in batch]),
        "sym_colours":  torch.stack([item[1]["sym_colours"]  for item in batch]),
        "sym_sizes":    torch.stack([item[1]["sym_sizes"]    for item in batch]),
        "text":         [item[1]["text"]                     for item in batch],
    }
    return images, labels


## 3. Data Loaders

In [3]:
IMG_SIZE      = 224
BATCH_SIZE    = 32
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = DiceDataset(DATASET_ROOT, "train", train_transform)
val_ds   = DiceDataset(DATASET_ROOT, "val",   eval_transform)
test_ds  = DiceDataset(DATASET_ROOT, "test",  eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=dice_collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=dice_collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=dice_collate_fn, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


imgs, labels = next(iter(train_loader))
print(f"\nImage batch      : {imgs.shape}")
print(f"sym_num_dice     : {labels['sym_num_dice'].shape}  sample={labels['sym_num_dice'][:4].tolist()}")
print(f"sym_values       : {labels['sym_values'].shape}    sample={labels['sym_values'][0].tolist()}")
print(f"sym_colours      : {labels['sym_colours'].shape}   sample={labels['sym_colours'][0].tolist()}")
print(f"sym_sizes        : {labels['sym_sizes'].shape}     sample={labels['sym_sizes'][0].tolist()}")
print(f"Example text     : {labels['text'][0]}")

counts = Counter(int(row["sym_num_dice"]) for row in train_ds.samples)
print(f"\nTrain class dist : { {DICE_CLASS_NAMES[k-1]: v for k, v in sorted(counts.items())} }")


DiceDataset (train): 4181 samples loaded.
DiceDataset (val): 899 samples loaded.
DiceDataset (test): 898 samples loaded.
Train batches: 131 | Val: 29 | Test: 29

Image batch      : torch.Size([32, 3, 224, 224])
sym_num_dice     : torch.Size([32])  sample=[2, 0, 0, 0]
sym_values       : torch.Size([32, 3])    sample=[5, 5, 3]
sym_colours      : torch.Size([32, 3])   sample=[1, 1, 1]
sym_sizes        : torch.Size([32, 3])     sample=[2, 1, 1]
Example text     : The scene contains a large red die showing six, a medium red die showing six and a medium red die showing four.

Train class dist : {'1 die': 1384, '2 dice': 1413, '3 dice': 1384}


## 4. Multi-Task Model Architecture

Each backbone feeds a **shared bottleneck**, then splits into **four parallel heads**:

| Head | Output shape | Predicts |
|---|---|---|
| `head_count`  | `[B, 3]` | number of dice (1/2/3) |
| `head_value`  | `[B, 3, 6]` | face value per slot (1–6 pips) |
| `head_colour` | `[B, 3, 7]` | colour per slot (7 options) |
| `head_size`   | `[B, 3, 3]` | size per slot (S/M/L) |

Loss = `λ_count·L_count + λ_value·L_value + λ_colour·L_colour + λ_size·L_size`


In [4]:
# ── CustomCNN backbone ────────────────────────────────────────────────
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32,  3, padding=1, bias=False), nn.BatchNorm2d(32),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,  64,  3, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.feature_dim = 256 * 4 * 4

    def forward(self, x):
        return self.features(x).flatten(1)


# ── ResNet-18 backbone ────────────────────────────────────────────────
def build_resnet18(pretrained: bool = True):
    m = models.resnet18(
        weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    )
    feat_dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, feat_dim


# ── EfficientNet-B0 backbone ──────────────────────────────────────────
def build_efficientnet_b0(pretrained: bool = True):
    m = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    )
    feat_dim = m.classifier[1].in_features
    m.classifier = nn.Identity()
    return m, feat_dim


# ── Four-task wrapper ─────────────────────────────────────────────────────
class MultiTaskDiceModel(nn.Module):
    """
    Shared backbone → shared bottleneck → four task heads:
      head_count  : [B, NUM_DICE_CLASSES]
      head_value  : [B, MAX_DICE, NUM_VALUE_CLASSES]
      head_colour : [B, MAX_DICE, NUM_COLOUR_CLASSES]
      head_size   : [B, MAX_DICE, NUM_SIZE_CLASSES]
    """
    def __init__(self, backbone: nn.Module, feature_dim: int, dropout: float = 0.4):
        super().__init__()
        self.backbone = backbone
        self.drop     = nn.Dropout(p=dropout)

        self.shared = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )

        self.head_count  = nn.Linear(512, NUM_DICE_CLASSES)
        self.head_value  = nn.Linear(512, NUM_VALUE_CLASSES  * MAX_DICE)
        self.head_colour = nn.Linear(512, NUM_COLOUR_CLASSES * MAX_DICE)
        self.head_size   = nn.Linear(512, NUM_SIZE_CLASSES   * MAX_DICE)

    def forward(self, x):
        feat   = self.drop(self.backbone(x))
        shared = self.shared(feat)
        return (
            self.head_count(shared),                                              # [B, 3]
            self.head_value(shared).view(-1, MAX_DICE, NUM_VALUE_CLASSES),        # [B, 3, 6]
            self.head_colour(shared).view(-1, MAX_DICE, NUM_COLOUR_CLASSES),      # [B, 3, 7]
            self.head_size(shared).view(-1, MAX_DICE, NUM_SIZE_CLASSES),          # [B, 3, 3]
        )


def make_model(name: str, pretrained: bool = True) -> MultiTaskDiceModel:
    if name == "CustomCNN":
        bb = CustomCNN(); feat_dim = bb.feature_dim
    elif name == "ResNet-18":
        bb, feat_dim = build_resnet18(pretrained)
    elif name == "EfficientNet-B0":
        bb, feat_dim = build_efficientnet_b0(pretrained)
    else:
        raise ValueError(f"Unknown model: {name}")
    return MultiTaskDiceModel(bb, feat_dim)


# Smoke test
_m = make_model("CustomCNN", pretrained=False)
_x = torch.randn(4, 3, 224, 224)
_lc, _lv, _lcol, _ls = _m(_x)
print(f"count  logits : {_lc.shape}    → expect [4, 3]")
print(f"value  logits : {_lv.shape}  → expect [4, 3, 6]")
print(f"colour logits : {_lcol.shape}  → expect [4, 3, 7]")
print(f"size   logits : {_ls.shape}    → expect [4, 3, 3]")
del _m, _x, _lc, _lv, _lcol, _ls


count  logits : torch.Size([4, 3])    → expect [4, 3]
value  logits : torch.Size([4, 3, 6])  → expect [4, 3, 6]
colour logits : torch.Size([4, 3, 7])  → expect [4, 3, 7]
size   logits : torch.Size([4, 3, 3])    → expect [4, 3, 3]


In [5]:
LAMBDA_COUNT  = 1.0
LAMBDA_VALUE  = 1.0
LAMBDA_COLOUR = 1.0
LAMBDA_SIZE   = 1.0

_ce = nn.CrossEntropyLoss()


def masked_slot_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    logits  : [B, MAX_DICE, C]
    targets : [B, MAX_DICE]  — -1 = absent slot (excluded from loss)
    """
    mask = targets >= 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)
    return _ce(logits[mask], targets[mask])


def multitask_loss(lc, lv, lcol, ls, labels):
    """Return (total, l_count, l_value, l_colour, l_size)."""
    l_count  = _ce(lc, labels["sym_num_dice"])
    l_value  = masked_slot_loss(lv,   labels["sym_values"])
    l_colour = masked_slot_loss(lcol, labels["sym_colours"])
    l_size   = masked_slot_loss(ls,   labels["sym_sizes"])
    total    = (LAMBDA_COUNT  * l_count  + LAMBDA_VALUE  * l_value +
                LAMBDA_COLOUR * l_colour + LAMBDA_SIZE   * l_size)
    return total, l_count, l_value, l_colour, l_size


# ── Prediction runner ─────────────────────────────────────────────────────
def get_predictions(model, loader):
    """Return dict of pred/true arrays for all four tasks plus avg loss."""
    model.eval()
    pc, tc, pv, tv, pcol, tcol, ps, ts = [], [], [], [], [], [], [], []
    total_loss, n = 0.0, 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            lbs  = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)
                    for k, v in labels.items()}
            lc, lv, lcol, ls = model(imgs)
            loss, *_ = multitask_loss(lc, lv, lcol, ls, lbs)
            total_loss += loss.item() * len(imgs); n += len(imgs)

            pc.extend(lc.argmax(1).cpu().numpy())
            tc.extend(lbs["sym_num_dice"].cpu().numpy())
            pv.append(lv.argmax(-1).cpu().numpy())
            tv.append(lbs["sym_values"].cpu().numpy())
            pcol.append(lcol.argmax(-1).cpu().numpy())
            tcol.append(lbs["sym_colours"].cpu().numpy())
            ps.append(ls.argmax(-1).cpu().numpy())
            ts.append(lbs["sym_sizes"].cpu().numpy())

    return {
        "pred_count":  np.array(pc),         "true_count":  np.array(tc),
        "pred_value":  np.vstack(pv),         "true_value":  np.vstack(tv),
        "pred_colour": np.vstack(pcol),        "true_colour": np.vstack(tcol),
        "pred_size":   np.vstack(ps),          "true_size":   np.vstack(ts),
        "avg_loss":    total_loss / n,
    }


def masked_metrics(true_arr, pred_arr, n_classes):
    """Flatten [N, MAX_DICE], drop pad slots (-1), return metric dict."""
    ft = true_arr.flatten(); fp = pred_arr.flatten()
    mask = ft >= 0; ft, fp = ft[mask], fp[mask]
    labs = list(range(n_classes))
    return {
        "accuracy":  round(accuracy_score(ft, fp), 4),
        "f1_macro":  round(f1_score(ft, fp, average="macro", zero_division=0, labels=labs), 4),
        "precision": round(precision_score(ft, fp, average="macro", zero_division=0, labels=labs), 4),
        "recall":    round(recall_score(ft, fp, average="macro", zero_division=0, labels=labs), 4),
        "cm":        confusion_matrix(ft, fp, labels=labs),
        "flat_true": ft, "flat_pred": fp,
    }


def compute_all_metrics(preds):
    return {
        "count":  {
            "accuracy":  round(accuracy_score(preds["true_count"], preds["pred_count"]), 4),
            "f1_macro":  round(f1_score(preds["true_count"], preds["pred_count"],
                               average="macro", zero_division=0), 4),
            "precision": round(precision_score(preds["true_count"], preds["pred_count"],
                               average="macro", zero_division=0), 4),
            "recall":    round(recall_score(preds["true_count"], preds["pred_count"],
                               average="macro", zero_division=0), 4),
            "cm":        confusion_matrix(preds["true_count"], preds["pred_count"], labels=[0,1,2]),
        },
        "value":  masked_metrics(preds["true_value"],  preds["pred_value"],  NUM_VALUE_CLASSES),
        "colour": masked_metrics(preds["true_colour"], preds["pred_colour"], NUM_COLOUR_CLASSES),
        "size":   masked_metrics(preds["true_size"],   preds["pred_size"],   NUM_SIZE_CLASSES),
    }


## 6. Training Loop

In [6]:
def train_model(model, name, epochs=30, lr=1e-3, weight_decay=1e-4):
    model     = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    best_state   = None
    epoch_log    = []
    t0           = time.time()

    print(f"\n{'='*68}")
    print(f"  Training : {name}  |  Epochs: {epochs}  |  LR: {lr}  |  Device: {DEVICE}")
    print(f"{'='*68}")
    print(f"{'Epoch':>6}  {'TrainAcc':>9}  {'ValAcc(cnt)':>11}  {'ValLoss':>9}")

    for ep in range(1, epochs + 1):
        model.train()
        correct, n = 0, 0
        for imgs, labels in train_loader:
            imgs = imgs.to(DEVICE)
            lbs  = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)
                    for k, v in labels.items()}
            optimizer.zero_grad()
            lc, lv, lcol, ls = model(imgs)
            loss, *_ = multitask_loss(lc, lv, lcol, ls, lbs)
            loss.backward()
            optimizer.step()
            correct += (lc.argmax(1) == lbs["sym_num_dice"]).sum().item()
            n       += len(imgs)
        train_acc = correct / n

        val_preds = get_predictions(model, val_loader)
        val_acc   = accuracy_score(val_preds["true_count"], val_preds["pred_count"])
        val_loss  = val_preds["avg_loss"]
        scheduler.step()

        epoch_log.append({"epoch": ep, "train_acc": round(train_acc,4),
                          "val_acc": round(val_acc,4), "val_loss": round(val_loss,4)})

        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if ep % 5 == 0 or ep == 1:
            print(f"{ep:>6}  {train_acc:>9.4f}  {val_acc:>11.4f}  {val_loss:>9.4f}")

    model.load_state_dict(best_state)
    test_preds   = get_predictions(model, test_loader)
    val_preds2   = get_predictions(model, val_loader)
    elapsed      = time.time() - t0
    test_metrics = compute_all_metrics(test_preds)
    val_metrics  = compute_all_metrics(val_preds2)

    print(f"\n  Best val acc (count) : {best_val_acc:.4f}   |   Time: {elapsed:.1f}s")
    for task, names in [("count", DICE_CLASS_NAMES), ("value", VALUE_CLASS_NAMES),
                         ("colour", COLOUR_CLASS_NAMES), ("size", SIZE_CLASS_NAMES)]:
        tm = test_metrics[task]
        print(f"  Test [{task:6s}]  Acc: {tm['accuracy']:.4f}   F1: {tm['f1_macro']:.4f}")

    # Per-class reports
    print("\n--- Count (test) ---")
    print(classification_report(test_preds["true_count"], test_preds["pred_count"],
                                target_names=DICE_CLASS_NAMES, digits=3))

    for task, class_names, n_cls, tk, pk in [
        ("Value",  VALUE_CLASS_NAMES,  NUM_VALUE_CLASSES,  "true_value",  "pred_value"),
        ("Colour", COLOUR_CLASS_NAMES, NUM_COLOUR_CLASSES, "true_colour", "pred_colour"),
        ("Size",   SIZE_CLASS_NAMES,   NUM_SIZE_CLASSES,   "true_size",   "pred_size"),
    ]:
        ft = test_preds[tk].flatten(); fp = test_preds[pk].flatten()
        mask = ft >= 0
        print(f"\n--- {task} (test, valid slots only) ---")
        print(classification_report(ft[mask], fp[mask],
                                    target_names=class_names,
                                    labels=list(range(n_cls)), digits=3))

    return {
        "log":          epoch_log,
        "val_metrics":  val_metrics,
        "test_metrics": test_metrics,
        "test_preds":   test_preds,
        "best_val_acc": best_val_acc,
        "train_time_s": elapsed,
    }


## 7. Run All Experiments

In [ ]:
torch.manual_seed(SEED)

experiments = [
    ("CustomCNN",       make_model("CustomCNN",       pretrained=False), 30, 1e-3, 1e-4),
    ("ResNet-18",       make_model("ResNet-18",        pretrained=True),  30, 5e-4, 1e-4),
    ("EfficientNet-B0", make_model("EfficientNet-B0",  pretrained=True),  30, 5e-4, 1e-4),
]

all_results = {}
for name, model, epochs, lr, wd in experiments:
    torch.manual_seed(SEED)
    result = train_model(model, name, epochs=epochs, lr=lr, weight_decay=wd)
    all_results[name] = result

    log_path = OUTPUT_DIR / f"multitask_log_{name.replace(' ','_').replace('-','')}.csv"
    pd.DataFrame(result["log"]).to_csv(log_path, index=False)
    print(f"  Epoch log → {log_path}\n")



  Training : CustomCNN  |  Epochs: 30  |  LR: 0.001  |  Device: cpu
 Epoch   TrainAcc  ValAcc(cnt)    ValLoss
     1     0.7965       0.9288     5.0805
     5     0.9629       0.7453     5.5041
    10     0.9790       1.0000     4.3439
    15     0.9835       1.0000     4.3984
    20     0.9912       1.0000     4.1345
    25     0.9950       1.0000     4.0928
    30     0.9950       1.0000     4.0809

  Best val acc (count) : 1.0000   |   Time: 10762.9s
  Test [count ]  Acc: 1.0000   F1: 1.0000
  Test [value ]  Acc: 0.4310   F1: 0.4074
  Test [colour]  Acc: 0.1403   F1: 0.1346
  Test [size  ]  Acc: 0.6066   F1: 0.5973

--- Count (test) ---
              precision    recall  f1-score   support

       1 die      1.000     1.000     1.000       302
      2 dice      1.000     1.000     1.000       308
      3 dice      1.000     1.000     1.000       288

    accuracy                          1.000       898
   macro avg      1.000     1.000     1.000       898
weighted avg      1.000  

## 8. Results Table — All Four Tasks

In [ ]:
rows = []
for name, result in all_results.items():
    tm = result["test_metrics"]
    rows.append({
        "Model":        name,
        "Pretrained":   "No" if name == "CustomCNN" else "Yes (ImageNet)",
        "Count Acc":    tm["count"]["accuracy"],   "Count F1":  tm["count"]["f1_macro"],
        "Value Acc":    tm["value"]["accuracy"],   "Value F1":  tm["value"]["f1_macro"],
        "Colour Acc":   tm["colour"]["accuracy"],  "Colour F1": tm["colour"]["f1_macro"],
        "Size Acc":     tm["size"]["accuracy"],    "Size F1":   tm["size"]["f1_macro"],
        "Train Time(s)": round(result["train_time_s"], 1),
    })

df = pd.DataFrame(rows)
print("=" * 100)
print("  MULTI-TASK RESULTS — TEST SET (Count · Value · Colour · Size)")
print("=" * 100)
print(df.to_string(index=False))

df.to_csv(OUTPUT_DIR / "multitask_results.csv", index=False)
print(f"\nSaved → {OUTPUT_DIR / 'multitask_results.csv'}")


In [ ]:
BG      = "#0f172a"
TEXT    = "#e2e8f0"
MUTED   = "#64748b"
PALETTE = {
    "CustomCNN":       "#3b82f6",
    "ResNet-18":       "#22c55e",
    "EfficientNet-B0": "#f59e0b",
}

TASKS = [
    ("count",  "Count (1/2/3)",     DICE_CLASS_NAMES,   None,             None),
    ("value",  "Face Value (1–6)",  VALUE_CLASS_NAMES,  "true_value",     "pred_value"),
    ("colour", "Colour (7 classes)",COLOUR_CLASS_NAMES, "true_colour",    "pred_colour"),
    ("size",   "Size (S/M/L)",      SIZE_CLASS_NAMES,   "true_size",      "pred_size"),
]

n_models, n_tasks = len(all_results), len(TASKS)
fig, axes = plt.subplots(n_models, n_tasks,
                         figsize=(5.5 * n_tasks, 4.5 * n_models),
                         facecolor=BG)
fig.suptitle("Multi-Task Confusion Matrices — Test Set  (Count · Value · Colour · Size)",
             color="white", fontsize=13, fontweight="bold", y=1.01)

for row_i, (model_name, result) in enumerate(all_results.items()):
    c     = PALETTE[model_name]
    preds = result["test_preds"]

    for col_j, (task_key, title, class_names, tk, pk) in enumerate(TASKS):
        ax = axes[row_i][col_j]

        if task_key == "count":
            y_true, y_pred = preds["true_count"], preds["pred_count"]
            labs = [0, 1, 2]
        else:
            ft = preds[tk].flatten(); fp = preds[pk].flatten()
            mask = ft >= 0; y_true, y_pred = ft[mask], fp[mask]
            labs = list(range(len(class_names)))

        cm_raw  = confusion_matrix(y_true, y_pred, labels=labs)
        cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True).clip(min=1)
        tm      = result["test_metrics"][task_key]

        sns.heatmap(
            cm_norm, ax=ax,
            cmap=sns.light_palette(c, as_cmap=True),
            annot=cm_raw, fmt="d",
            linewidths=0.4, linecolor="#334155",
            xticklabels=class_names, yticklabels=class_names,
            cbar=False,
            annot_kws={"size": 8, "weight": "bold", "color": "white"},
        )
        ax.set_facecolor(BG)
        ax.set_title(
            f"{model_name} | {title}\nAcc {tm['accuracy']:.3f}   F1 {tm['f1_macro']:.3f}",
            color=c, fontsize=9, fontweight="bold", pad=7
        )
        ax.set_xlabel("Predicted", color=MUTED, fontsize=7)
        ax.set_ylabel("True",      color=MUTED, fontsize=7)
        ax.tick_params(colors=MUTED, labelsize=6)
        for sp in ax.spines.values(): sp.set_edgecolor("#334155")

plt.tight_layout()
cm_path = OUTPUT_DIR / "multitask_confusion_matrices.png"
fig.savefig(cm_path, dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {cm_path}")


## 10. Learning Curves & Comparative Bar Charts

In [ ]:
fig = plt.figure(figsize=(19, 12), facecolor=BG)
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.48, wspace=0.33,
                         left=0.05, right=0.97, top=0.88, bottom=0.09)

def style_ax(ax):
    ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_edgecolor("#334155")
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.yaxis.grid(True, color="#1e293b", linestyle="--", lw=0.6)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)

# Row 0: per-model learning curves (count-task val accuracy)
for col, (name, result) in enumerate(all_results.items()):
    ax  = fig.add_subplot(gs[0, col]); style_ax(ax)
    log = result["log"]
    ep  = [r["epoch"] for r in log]
    c   = PALETTE[name]
    ax.plot(ep, [r["train_acc"] for r in log], color=c, lw=2,             label="Train acc")
    ax.plot(ep, [r["val_acc"]   for r in log], color=c, lw=2, ls="--", alpha=0.7, label="Val acc")
    ax.axhline(result["test_metrics"]["count"]["accuracy"],
               color="#ef4444", lw=1.2, ls=":", label=f"Test {result['test_metrics']['count']['accuracy']:.3f}")
    ax.set_ylim(0.2, 1.10)
    ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel("Count Accuracy", fontsize=9)
    ax.set_title(name, color=c, fontsize=11, fontweight="bold", pad=7)
    ax.legend(fontsize=7, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

# Row 0 col 3: F1 heatmap across models × tasks
ax_h = fig.add_subplot(gs[0, 3]); style_ax(ax_h)
task_keys   = ["count", "value", "colour", "size"]
task_labels = ["Count", "Value", "Colour", "Size"]
model_names = list(all_results.keys())
f1_matrix   = np.array([[all_results[m]["test_metrics"][t]["f1_macro"]
                          for t in task_keys] for m in model_names])
sns.heatmap(f1_matrix, ax=ax_h,
            cmap="YlGn", vmin=0, vmax=1,
            annot=True, fmt=".3f", annot_kws={"size": 9, "weight": "bold"},
            xticklabels=task_labels, yticklabels=model_names,
            linewidths=0.5, linecolor="#334155", cbar=False)
ax_h.set_facecolor(BG)
ax_h.set_title("F1 (macro) — All Tasks", color=TEXT, fontsize=10, fontweight="bold", pad=7)
ax_h.tick_params(colors=MUTED, labelsize=8)
for sp in ax_h.spines.values(): sp.set_edgecolor("#334155")

# Row 1: grouped bar chart per task (Accuracy)
for col, (task_key, task_label) in enumerate(zip(task_keys, task_labels)):
    ax = fig.add_subplot(gs[1, col]); style_ax(ax)
    model_list = list(all_results.keys())
    x = np.arange(len(model_list)); w = 0.35
    accs = [all_results[m]["test_metrics"][task_key]["accuracy"] for m in model_list]
    f1s  = [all_results[m]["test_metrics"][task_key]["f1_macro"]  for m in model_list]
    bars_acc = ax.bar(x - w/2, accs, w, label="Accuracy", zorder=3,
                      color=[PALETTE[m] for m in model_list], alpha=0.9)
    bars_f1  = ax.bar(x + w/2, f1s,  w, label="F1 macro", zorder=3,
                      color=[PALETTE[m] for m in model_list], alpha=0.5)
    for bar, v in list(zip(bars_acc, accs)) + list(zip(bars_f1, f1s)):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.2f}",
                ha="center", va="bottom", color="white", fontsize=6.5, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("EfficientNet-", "Eff-") for m in model_list],
                       fontsize=7, color=MUTED)
    ax.set_ylim(0, 1.22); ax.set_ylabel("Score", fontsize=8)
    ax.set_title(f"{task_label} Task — Test Set", color=TEXT, fontsize=10, fontweight="bold", pad=7)
    if col == 0:
        ax.legend(fontsize=7, facecolor="#1e293b", edgecolor="#334155", labelcolor="#cbd5e1")

fig.suptitle("Multi-Task Dice — Count · Face Value · Colour · Size  |  CustomCNN vs ResNet-18 vs EfficientNet-B0",
             color="white", fontsize=12, fontweight="bold", y=0.96)
curves_path = OUTPUT_DIR / "multitask_learning_curves.png"
fig.savefig(curves_path, dpi=130, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved → {curves_path}")
